<a href="https://colab.research.google.com/github/kodomotachi/heartify-AI/blob/main/AgentExercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install -U opik langgraph langchain langchain-groq langchain-pinecone langchain-huggingface sentence-transformers pinecone-client

In [ ]:
# Library Loading
import os
from typing import TypedDict, List, Dict, Any
from google.colab import userdata

# LangChain Core & Groq
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import JsonOutputParser, StrOutputParser # Gom lại

# Vector DB & Embeddings
from pinecone import Pinecone, ServerlessSpec # Dùng để quản lý Index (Admin tasks)
from langchain_pinecone import PineconeVectorStore # Dùng để Search/Retriever
from sentence_transformers import SentenceTransformer
from langchain_huggingface import HuggingFaceEmbeddings

# LangGraph & Monitoring
from langgraph.graph import StateGraph, START, END
from opik.integrations.langchain import OpikTracer

import hashlib

In [ ]:
# Environment Loading
def load_env_var(key, required=True):
    """Loads a key from Colab userdata to os.environ."""
    try:
        os.environ[key] = userdata.get(key)
        return True
    except Exception:
        if required:
            print(f"Error: Required secret '{key}' not found.")
        return False

# Configure standard keys
load_env_var("GROQ_API_KEY")
load_env_var("PINECONE_API_KEY")

# Configure Opik for observability
if load_env_var("OPIK_API_KEY", required=False):
    os.environ["OPIK_WORKSPACE"] = "phuc-duy-loc-nguyen" # Optional: Change if you have a specific workspace
    os.environ["OPIK_PROJECT_NAME"] = "Exercise"
    print("Opik configured successfully.")
else:
    print("Warning: OPIK_API_KEY not found in secrets. Tracing might not work.")

Opik configured successfully.


## Nghiên cứu nguyên tắc tập luyện



In [ ]:
# Create the workout_guidelines dictionary based on heart risk zones
workout_guidelines = {
    'Alert': {
        'risk_level': 'High Risk',
        'guidelines': 'Very low intensity, short duration (max 15 mins). Focus on breathing or gentle stretching. Avoid heart rate spikes.'
    },
    'Care': {
        'risk_level': 'Medium Risk',
        'guidelines': 'Moderate intensity, medium duration (20-30 mins). Warm-up and cool-down are essential.'
    },
    'Healthy': {
        'risk_level': 'Low Risk',
        'guidelines': 'Standard to high intensity allowed, flexible duration. Mix of strength and cardio based on preferences.'
    }
}

# Print the dictionary to confirm structure
import pprint
pprint.pprint(workout_guidelines)

{'Alert': {'guidelines': 'Very low intensity, short duration (max 15 mins). '
                         'Focus on breathing or gentle stretching. Avoid heart '
                         'rate spikes.',
           'risk_level': 'High Risk'},
 'Care': {'guidelines': 'Moderate intensity, medium duration (20-30 mins). '
                        'Warm-up and cool-down are essential.',
          'risk_level': 'Medium Risk'},
 'Healthy': {'guidelines': 'Standard to high intensity allowed, flexible '
                           'duration. Mix of strength and cardio based on '
                           'preferences.',
             'risk_level': 'Low Risk'}}


## Thiết lập State cho LangGraph

Define the Graph State structure using TypedDict to manage data flow (inputs: Zone, Energy, Preferences; outputs: Intensity, Search Query, Retrieved Exercises, Final Plan).


In [ ]:
# WorkoutState

from typing import TypedDict, List, Any, Dict

class WorkoutState(TypedDict):
    # Inputs
    zone: str                           # Heart risk zone
    energy_level: str                   # User energy level
    preferences: str                    # User preferences
    risk_factors: List[str]             # Specific risk factors
    biometrics: str                     # User height, weight, BMI info
    limitations: str                    # Physical limitations (e.g., injuries)

    # Intermediate - Intensity
    intensity: dict                     # Derived intensity guidelines

    # Intermediate - Query
    search_query: str                   # Semantic query string
    filter_params: Dict[str, Any]       # Pinecone metadata filters
    intensity_range: Dict[str, Any]     # For debugging

    # Intermediate - Retrieval
    retrieved_exercises: List[dict]     # Retrieved exercises

    # Intermediate - References
    references: List[Dict[str, Any]]    # NEW: Structured references with hash

    # Output
    final_plan: str                     # Final workout recommendation

print(WorkoutState.__annotations__)

{'zone': <class 'str'>, 'energy_level': <class 'str'>, 'preferences': <class 'str'>, 'risk_factors': typing.List[str], 'biometrics': <class 'str'>, 'limitations': <class 'str'>, 'intensity': <class 'dict'>, 'search_query': <class 'str'>, 'filter_params': typing.Dict[str, typing.Any], 'intensity_range': typing.Dict[str, typing.Any], 'retrieved_exercises': typing.List[dict], 'references': typing.List[typing.Dict[str, typing.Any]], 'final_plan': <class 'str'>}


## Xây dựng Node xác định cường độ (Intensity Node)

Develop the logic for the `determine_intensity` node to derive workout intensity guidelines based on the user's risk zone.


In [ ]:
def determine_intensity(state: WorkoutState) -> dict:
    """
    Node to determine workout intensity based on the heart risk zone.
    Accesses the global 'workout_guidelines' dictionary.
    """
    zone = state.get('zone')

    # Retrieve guidelines, defaulting to 'Alert' (High Risk) if zone is missing or invalid for safety
    guidelines = workout_guidelines.get(zone, workout_guidelines['Alert'])

    return {"intensity": guidelines}

# Test the function with a sample state
sample_state = {"zone": "Care", "energy_level": "Medium", "preferences": "Yoga"}
intensity_result = determine_intensity(sample_state)

print("Intensity Node Output:")
import pprint
pprint.pprint(intensity_result)

Intensity Node Output:
{'intensity': {'guidelines': 'Moderate intensity, medium duration (20-30 '
                             'mins). Warm-up and cool-down are essential.',
               'risk_level': 'Medium Risk'}}


## Xây dựng Node tạo truy vấn (Query Node)

Create the `generate_query` node logic to construct a search query for the vector database by combining workout intensity guidelines, user energy level, and preferences.


In [ ]:
# Generate Query Node (Enhanced)

def generate_query(state: WorkoutState) -> dict:
    """
    Node tạo query thông minh cho Pinecone.

    Tạo 2 loại query:
    1. semantic_query: Cho semantic search
    2. filter_params: Cho metadata filtering
    """
    intensity_data = state.get('intensity', {})
    guidelines = intensity_data.get('guidelines', 'General workout')
    risk_level = intensity_data.get('risk_level', 'Unknown Risk')
    energy = state.get('energy_level', 'Medium')
    preferences = state.get('preferences', 'General')
    zone = state.get('zone', 'Healthy')

    # ============ 1. MAP ZONE -> INTENSITY FILTERS ============
    # Mapping từ zone sang intensity score range
    intensity_mapping = {
        'Alert': {
            'min': 1,
            'max': 2,
            'labels': ['VERY_LOW', 'LOW'],
            'keywords': 'gentle low-impact breathing stretching relaxation'
        },
        'Care': {
            'min': 2,
            'max': 3,
            'labels': ['LOW', 'MODERATE'],
            'keywords': 'moderate controlled steady sustainable'
        },
        'Healthy': {
            'min': 1,
            'max': 5,
            'labels': ['VERY_LOW', 'LOW', 'MODERATE', 'HIGH', 'VERY_HIGH'],
            'keywords': 'any intensity strength power endurance'
        }
    }

    zone_config = intensity_mapping.get(zone, intensity_mapping['Alert'])

    # ============ 2. MAP ENERGY -> EQUIPMENT PREFERENCE ============
    energy_equipment_map = {
        'Low': 'body weight',      # Không dùng thiết bị
        'Medium': None,            # Linh hoạt
        'High': None               # Linh hoạt
    }

    preferred_equipment = energy_equipment_map.get(energy)

    # ============ 3. BUILD SEMANTIC QUERY ============
    # Query giống với embedding_text structure
    semantic_query = f"""
    Exercise for: {preferences}
    Energy level: {energy}
    Intensity: {zone_config['keywords']}
    Target: workout suitable for {risk_level} focusing on {preferences}
    """.strip()

    # ============ 4. BUILD METADATA FILTERS ============
    filter_dict = {
        'intensity_score': {
            '$gte': zone_config['min'],
            '$lte': zone_config['max']
        }
    }

    # Add equipment filter nếu cần
    if preferred_equipment:
        filter_dict['equipment'] = {'$eq': preferred_equipment}

    # Add body_part filter nếu preferences cụ thể
    body_part_keywords = {
        'chest': 'chest',
        'back': 'back',
        'legs': 'legs',
        'arms': 'upper arms',
        'shoulders': 'shoulders',
        'cardio': 'cardio',
        'abs': 'waist',
        'core': 'waist'
    }

    pref_lower = preferences.lower()
    for keyword, body_part in body_part_keywords.items():
        if keyword in pref_lower:
            filter_dict['body_part'] = {'$eq': body_part}
            break

    # ============ 5. RETURN STATE UPDATE ============
    return {
        "search_query": semantic_query,
        "filter_params": filter_dict,  # Thêm field mới
        "intensity_range": zone_config  # Để debug
    }


# Test the enhanced query generation
sample_state = {
    "zone": "Care",
    "energy_level": "Medium",
    "preferences": "chest exercises"
}
intensity_result = determine_intensity(sample_state)
state_with_intensity = {**sample_state, **intensity_result}
query_result = generate_query(state_with_intensity)

print("Enhanced Query Node Output:")
import pprint
pprint.pprint(query_result)

Enhanced Query Node Output:
{'filter_params': {'body_part': {'$eq': 'chest'},
                   'intensity_score': {'$gte': 2, '$lte': 3}},
 'intensity_range': {'keywords': 'moderate controlled steady sustainable',
                     'labels': ['LOW', 'MODERATE'],
                     'max': 3,
                     'min': 2},
 'search_query': 'Exercise for: chest exercises\n'
                 '    Energy level: Medium\n'
                 '    Intensity: moderate controlled steady sustainable\n'
                 '    Target: workout suitable for Medium Risk focusing on '
                 'chest exercises'}


## Tích hợp Pinecone (Retrieval Node)

Develop the `retrieve_exercises` node logic. To allow the pipeline to run without external API keys, implement a simulated retrieval function that mimics querying a vector database to return relevant exercises based on the search query.


In [ ]:
# Initialize Pinecone
PINECONE_API_KEY = userdata.get('PINECONE_API_KEY')
PINECONE_INDEX_NAME = "fitness-exercises"  # Tên index bạn đã tạo

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(PINECONE_INDEX_NAME)

# Load embedding model (PHẢI CÙNG MODEL với lúc indexing)
embedding_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print(f"Pinecone connected: {index.describe_index_stats()}")
print(f"Embedding model loaded: {embedding_model}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Pinecone connected: {'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1324}},
 'total_vector_count': 1324,
 'vector_type': 'dense'}
Embedding model loaded: SentenceTransformer(
  (0): Transformer({'max_seq_length': 256, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 384, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
)


In [ ]:
# Retrieve Node with Real Pinecone Integration

def retrieve_exercises(state: WorkoutState) -> dict:
    """
    Node retrieve exercises từ Pinecone với semantic search + metadata filtering.

    Returns:
        dict với key 'retrieved_exercises': List[dict]
    """
    # Extract query components
    semantic_query = state.get('search_query', '')
    filter_params = state.get('filter_params', {})

    print(f"\n{'='*60}")
    print(f"PINECONE RETRIEVAL")
    print(f"{'='*60}")
    print(f"Semantic Query: {semantic_query[:100]}...")
    print(f"Filters: {filter_params}")

    try:
        # ============ 1. ENCODE QUERY ============
        query_embedding = embedding_model.encode(
            semantic_query,
            normalize_embeddings=True  # Phải giống lúc indexing
        ).tolist()

        print(f"Query embedded: {len(query_embedding)}D vector")

        # ============ 2. QUERY PINECONE ============
        results = index.query(
            vector=query_embedding,
            top_k=10,  # Lấy 10 exercises tốt nhất
            filter=filter_params,
            include_metadata=True
        )

        print(f"Found {len(results['matches'])} exercises")

        # ============ 3. FORMAT RESULTS ============
        retrieved_exercises = []

        for i, match in enumerate(results['matches'], 1):
            metadata = match['metadata']

            exercise = {
                'rank': i,
                'name': metadata.get('exercise_name', 'Unknown'),
                'target_muscle': metadata.get('target_muscle', 'N/A'),
                'body_part': metadata.get('body_part', 'N/A'),
                'equipment': metadata.get('equipment', 'N/A'),
                'intensity_label': metadata.get('intensity_label', 'N/A'),
                'intensity_score': metadata.get('intensity_score', 0),
                'secondary_muscles': metadata.get('secondary_muscles', 'N/A'),
                'instructions': metadata.get('full_instructions', 'N/A'),
                'gif_url': metadata.get('gif_url', 'N/A'), # NEW: Extract GIF URL
                'similarity_score': round(match['score'], 3)
            }

            retrieved_exercises.append(exercise)

            # Print top 3 for verification
            if i <= 3:
                print(f"\n{i}. {exercise['name']}")
                print(f"   Target: {exercise['target_muscle']} | Equipment: {exercise['equipment']}")
                print(f"   Intensity: {exercise['intensity_label']} ({exercise['intensity_score']}/5)")
                print(f"   Similarity: {exercise['similarity_score']}")

        print(f"{'='*60}\n")

        return {"retrieved_exercises": retrieved_exercises}

    except Exception as e:
        print(f"Error querying Pinecone: {str(e)}")

        # Fallback: Return mock data
        print("Using fallback mock data...")
        return {
            "retrieved_exercises": [
                {
                    'rank': 1,
                    'name': 'Fallback Exercise',
                    'target_muscle': 'general',
                    'body_part': 'full body',
                    'equipment': 'body weight',
                    'intensity_label': 'MODERATE',
                    'intensity_score': 3,
                    'secondary_muscles': 'N/A',
                    'instructions': 'Exercise data unavailable',
                    'gif_url': 'N/A',
                    'similarity_score': 0.0
                }
            ]
        }

## Xây dựng Node tạo gợi ý (Generation Node)

Develop the `synthesize_plan` node logic to generate the final workout recommendation. To ensure execution without an LLM API key, implement a rule-based function that formats the retrieved exercises and guidelines into a coherent plan.


In [ ]:
# Cell: Enhanced Generation Node
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
import hashlib

def synthesize_plan(state: WorkoutState) -> dict:
    """
    Node synthesize workout plan với Groq LLM.
    Sử dụng retrieved exercises từ Pinecone.
    """

    client = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)

    # ============ EXTRACT CONTEXT ============
    zone = state.get('zone', 'Unknown')
    energy = state.get('energy_level', 'Medium')
    preferences = state.get('preferences', 'General')
    risk_factors = state.get('risk_factors', [])
    biometrics = state.get('biometrics', 'Not available')
    limitations = state.get('limitations', 'None')

    risk_factors_str = ", ".join(risk_factors) if risk_factors else "None"

    intensity_data = state.get('intensity', {})
    guidelines = intensity_data.get('guidelines', 'Exercise with caution.')
    risk_level = intensity_data.get('risk_level', 'Unknown Risk')

    # NEW: Format exercises từ Pinecone
    exercises = state.get('retrieved_exercises', [])

    # === PROCESS REFERENCES ===
    references = []
    ref_text = "\n\n--- **Reference Links** ---\n"

    if not exercises:
        exercises_str = "No exercises found."
    else:
        # Format for LLM
        exercises_str = "\n\n".join([
            f"{ex['rank']}. **{ex['name']}** (Similarity: {ex['similarity_score']})\n"
            f"   - Target: {ex['target_muscle']}\n"
            f"   - Equipment: {ex['equipment']}\n"
            f"   - Intensity: {ex['intensity_label']} ({ex['intensity_score']}/5)\n"
            f"   - Secondary: {ex['secondary_muscles']}\n"
            f"   - Instructions: {ex['instructions'][:200]}..."  # Truncate long instructions
            for ex in exercises[:5]  # Top 5 exercises for plan
        ])

        # Format References (Top 5)
        for i, ex in enumerate(exercises[:5], 1):
            url = ex.get('gif_url', 'N/A')
            if url and url != 'N/A':
                # Create SHA256 Hash (16 chars)
                hash_id = hashlib.sha256(url.encode()).hexdigest()[:16]

                references.append({
                    "index": i,
                    "name": ex['name'],
                    "url": url,
                    "hash_id": hash_id
                })

                # Append to reference text
                ref_text += f"[{i}] {ex['name'].title()}: {url}\n"

    # ============ CONSTRUCT PROMPT ============
    prompt_template = PromptTemplate.from_template(
        """
You are a professional fitness trainer and medical assistant.

Create a personalized, safe, and effective workout plan for a user with the following profile:

**User Profile:**
- Heart Risk Zone: {zone} ({risk_level})
- Risk Factors: {risk_factors_str}
- Physical Limitations: {limitations}
- Biometrics: {biometrics}
- Energy Level: {energy}
- Preferences: {preferences}

**Medical Guidelines (STRICTLY FOLLOW):**
{guidelines}

**Available Exercises (Retrieved from Vector Database):**
{exercises_str}

**Instructions:**
1. Design a complete workout including warm-up, main workout, and cool-down
2. Select exercises from the available list that match the medical guidelines
3. PAY CLOSE ATTENTION to the Risk Factors ({risk_factors_str}) and Limitations ({limitations}) when selecting exercises.
4. Ensure total duration aligns with the guidelines
5. Provide specific sets/reps/duration for each exercise
6. Add safety tips and modifications if needed
7. Make the plan engaging and motivating

Generate the workout plan now:
        """
    )

    chain = prompt_template | client

    try:
        response = chain.invoke({
            "zone": zone,
            "risk_level": risk_level,
            "risk_factors_str": risk_factors_str,
            "limitations": limitations,
            "biometrics": biometrics,
            "energy": energy,
            "preferences": preferences,
            "guidelines": guidelines,
            "exercises_str": exercises_str
        })
        final_plan = response.content

        # Append references to the plan
        if references:
            final_plan += ref_text

    except Exception as e:
        final_plan = f"Error calling Groq API: {str(e)}"

    return {"final_plan": final_plan, "references": references}

## Hoàn thiện Pipeline LangGraph

Construct the LangGraph workflow by connecting the previously defined nodes and testing the complete execution flow.


In [ ]:
# Cell: Process Backend Data & Run Pipeline

from langgraph.graph import StateGraph, END

# 1. Recompile the Workflow
workflow = StateGraph(WorkoutState)
workflow.add_node("intensity_node", determine_intensity)
workflow.add_node("query_node", generate_query)
workflow.add_node("retrieval_node", retrieve_exercises)
workflow.add_node("generation_node", synthesize_plan)

workflow.set_entry_point("intensity_node")
workflow.add_edge("intensity_node", "query_node")
workflow.add_edge("query_node", "retrieval_node")
workflow.add_edge("retrieval_node", "generation_node")
workflow.add_edge("generation_node", END)

app = workflow.compile()

# 2. Define Backend Input Data (FULL JSON)
backend_data = {
  "user_health_record": {
    "userId": "550e8400-e29b-41d4-a716-446655440000",
    "recordedAt": "2023-10-27T10:00:00.000Z",
    "ageAtRecord": 45,
    "systolicBp": 145,
    "diastolicBp": 90,
    "totalCholesterol": 240.50,
    "hdlCholesterol": 35.00,
    "isSmoker": True,
    "isDiabetic": False,
    "isTreatedHypertension": True,
    "measurements": {
      "height": 175,
      "weight": 82,
      "bmi": 26.7,
      "waistCircumference": 98
    },
    "riskLevel": "high",
    "riskScore": 14.5000,
    "riskPercentage": 18.25,
    "riskAlgorithm": "framingham",
    "identifiedRiskFactors": [
      "smoker",
      "high_systolic_bp",
      "low_hdl",
      "high_cholesterol"
    ]
  },
  "user_preferences": {
    "id": "a0eebc99-9c0b-4ef8-bb6d-6bb9bd380a11",
    "userId": "b1f4cc88-8d1c-4ef8-aa6d-6bb9bd380b22",
    "dateOfBirth": "1995-08-15",
    "gender": "MALE",
    "country": "VNM",
    "latestMeasurements": {
      "weight": {
        "value": 70.5,
        "unit": "kg"
      },
      "height": {
        "value": 175,
        "unit": "cm"
      },
      "bmi": 23.02
    },
    "allergies": {
      "options": ["Peanuts", "Seafood"],
      "details": "Severe reaction to shrimp"
    },
    "medicalConditions": {
      "options": ["Hypertension"],
      "details": "Diagnosed in 2023, currently managed"
    },
    "medications": {
      "options": ["Amlodipine"],
      "details": "5mg daily in the morning"
    },
    "physicalLimitations": {
      "options": ["Knee Injury"],
      "details": "Avoid heavy squats"
    },
    "createdAt": "2024-02-06T05:00:00.000Z",
    "updatedAt": "2024-02-06T05:00:00.000Z"
  }
}

# 3. Parse and Map Backend Data
health_record = backend_data["user_health_record"]
preferences_data = backend_data.get("user_preferences", {})

# --- A. Biometrics ---
# Prefer latest measurements from preferences if available, else fall back to health record
latest_measurements = preferences_data.get("latestMeasurements", {})
if latest_measurements:
    height = latest_measurements.get("height", {}).get("value", "N/A")
    weight = latest_measurements.get("weight", {}).get("value", "N/A")
    bmi = latest_measurements.get("bmi", "N/A")
    biometrics_src = "Latest (User Prefs)"
else:
    hr_measurements = health_record.get("measurements", {})
    height = hr_measurements.get("height", "N/A")
    weight = hr_measurements.get("weight", "N/A")
    bmi = hr_measurements.get("bmi", "N/A")
    biometrics_src = "Health Record"

biometrics_str = f"Height: {height}, Weight: {weight}, BMI: {bmi} (Source: {biometrics_src})"

# --- B. Risk & Zone ---
# Always use the medical risk level for safety
risk_map = {"high": "Alert", "medium": "Care", "low": "Healthy"}
risk_level_input = health_record.get("riskLevel", "medium").lower()
zone = risk_map.get(risk_level_input, "Care")

# --- C. Risk Factors & Conditions ---
risk_factors = health_record.get("identifiedRiskFactors", [])
# Add medical conditions from preferences
med_conditions = preferences_data.get("medicalConditions", {})
if med_conditions:
    cond_list = med_conditions.get("options", [])
    cond_details = med_conditions.get("details", "")
    risk_factors.extend([f"Condition: {c}" for c in cond_list])

# --- D. Limitations ---
limitations_data = preferences_data.get("physicalLimitations", {})
limitations_str = "None"
if limitations_data:
    opts = ", ".join(limitations_data.get("options", []))
    dets = limitations_data.get("details", "")
    limitations_str = f"{opts} - {dets}".strip(" - ")

# --- E. Preferences ---
# Construct a preference string for the query
prefs_context = "General Health"
# Example: combining gender and potential goals if available (not in this JSON, so generic)
prefs_context = f"User Gender: {preferences_data.get('gender', 'N/A')}"

# --- F. Energy ---
energy_level = "Low" if zone == "Alert" else "Medium"


# 4. Prepare Graph Inputs
inputs = {
    "zone": zone,
    "energy_level": energy_level,
    "preferences": prefs_context,
    "risk_factors": risk_factors,
    "biometrics": biometrics_str,
    "limitations": limitations_str
}

print(f"Processing Input for User {backend_data['user_health_record']['userId']}...")
print(f"Zone: {zone}")
print(f"Biometrics: {biometrics_str}")
print(f"Risk Factors: {risk_factors}")
print(f"Limitations: {limitations_str}")
print("="*60)
# Initialize Opik Tracer
opik_tracer = OpikTracer(graph=app.get_graph(xray=True))

# 5. Execute Pipeline
result = app.invoke(inputs, config={"callbacks": [opik_tracer]})

print("\nFINAL WORKOUT RECOMMENDATION:\n")
print(result['final_plan'])

OPIK: Started logging traces to the "Exercise" project at https://www.comet.com/opik/api/v1/session/redirect/projects/?trace_id=019c3b8e-8e35-713c-ad68-06e8b89a7d6a&path=aHR0cHM6Ly93d3cuY29tZXQuY29tL29waWsvYXBpLw==.


Processing Input for User 550e8400-e29b-41d4-a716-446655440000...
Zone: Alert
Biometrics: Height: 175, Weight: 70.5, BMI: 23.02 (Source: Latest (User Prefs))
Risk Factors: ['smoker', 'high_systolic_bp', 'low_hdl', 'high_cholesterol', 'Condition: Hypertension']
Limitations: Knee Injury - Avoid heavy squats

PINECONE RETRIEVAL
Semantic Query: Exercise for: User Gender: MALE
    Energy level: Low
    Intensity: gentle low-impact breathing str...
Filters: {'intensity_score': {'$gte': 1, '$lte': 2}, 'equipment': {'$eq': 'body weight'}}
Query embedded: 384D vector
Found 10 exercises

1. dynamic chest stretch (male)
   Target: pectorals | Equipment: body weight
   Intensity: VERY_LOW (1.0/5)
   Similarity: 0.52

2. runners stretch
   Target: hamstrings | Equipment: body weight
   Intensity: VERY_LOW (1.0/5)
   Similarity: 0.502

3. world greatest stretch
   Target: hamstrings | Equipment: body weight
   Intensity: VERY_LOW (1.0/5)
   Similarity: 0.488


FINAL WORKOUT RECOMMENDATION:

**Person

In [ ]:
result['references']

[{'index': 1,
  'name': 'dynamic chest stretch (male)',
  'url': 'https://v2.exercisedb.io/image/J3JL-jzjKUWoGX',
  'hash_id': 'faeed59e810f70d8'},
 {'index': 2,
  'name': 'runners stretch',
  'url': 'https://v2.exercisedb.io/image/dA2RZTTki9RNZc',
  'hash_id': '6475cbd8e49970f2'},
 {'index': 3,
  'name': 'world greatest stretch',
  'url': 'https://v2.exercisedb.io/image/3A5fPe5oCr2EwN',
  'hash_id': 'e97ec115102136c1'},
 {'index': 4,
  'name': 'seated calf stretch (male)',
  'url': 'https://v2.exercisedb.io/image/LMbVbz35RTzTWa',
  'hash_id': '8a644c10f1483b7e'},
 {'index': 5,
  'name': 'two toe touch (male)',
  'url': 'https://v2.exercisedb.io/image/ONl1K6pDfoHBXG',
  'hash_id': '82cbd44a393ffe53'}]

In [ ]:
## SHOW PINECONE SAMPLE
# 1. Get Index Statistics
stats = index.describe_index_stats()
print("--- Index Statistics ---")
print(stats)

# 2. Infer Schema from a Sample Record
# Generate a dummy embedding using the loaded model
sample_embedding = embedding_model.encode("nutrition sample").tolist()

# Query for 1 item to inspect metadata
sample_result = index.query(
    vector=sample_embedding,
    top_k=1,
    include_metadata=True
)

print("\n--- Inferred Metadata Schema ---")
if sample_result['matches']:
    metadata = sample_result['matches'][0]['metadata']
    print(f"Sample Item: {sample_result['matches'][0]['id']}")
    for key, value in metadata.items():
        print(f"- {key}: {type(value).__name__} (e.g., {value})")
else:
    print("Index appears to be empty, cannot infer schema.")

--- Index Statistics ---
{'dimension': 384,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'': {'vector_count': 1324}},
 'total_vector_count': 1324,
 'vector_type': 'dense'}

--- Inferred Metadata Schema ---
Sample Item: 3300
- body_part: str (e.g., waist)
- equipment: str (e.g., body weight)
- exercise_name: str (e.g., lean planche)
- full_instructions: str (e.g., Exercise: lean planche

Target Muscle: abs
Body Part: waist
Equipment Needed: body weight
Intensity Level: MODERATE
Visual Demonstration Available: Yes
Secondary Muscles: shoulders, chest

How to Perform:
Step 1: Start in a push-up position with your hands shoulder-width apart and your body straight. Step 2: Engage your core and slowly shift your weight forward, bringing your shoulders past your hands. Step 3: Keep your elbows slightly bent and your body straight as you lean forward. Step 4: Hold this position for a few seconds, then slowly return to the starting position. Step 5: Repeat for the desired number 